# 🧪 CoChem Ecosystem: Master Orchestrator

Welcome to the definitive graphical entry point for the CoChem computational chemistry pipeline. This notebook (`Start_Here.ipynb`) connects the interactive front-end dashboards to the hardened, registry-first **CoChem-CORE** operating system.

---

### 🔒 Strict Filesystem Air-Gap Enforced
To prevent repository bloat and ensure FERPA/proprietary data compliance, all execution logic remains locked in the static `$HOME/CoChem-CORE/` tier. All dynamically generated trajectories, HDF5 databases (`landscape.h5`), and ORCA wavefunctions (`.gbw`) are automatically routed to the isolated `$HOME/CoChem_Artifacts/` volume.

### 🚀 Initialization Sequence
* **Phases 1-4:** OS Profiling, Memory Scaling, & Micro-Silo Assembly
* **Phases 5-7:** Engine Cryptographic Validation (ORCA, OpenMPI, g-xTB)
* **Phases 10-11:** Physical Constant Locking & `cochem_system_config.json` Registry Generation
* **Handoff:** Automatic rendering of the **CoChem-UNITY** GUI for module deployment.

---

### ⚠️ ACTION REQUIRED: Kernel Selection & Execution

Before proceeding, you must ensure this notebook is attached to the correct Python kernel so the orchestrator can securely build the environment.

**Step 1: Select the Kernel**
1. Look at the **top right corner** of your Jupyter interface.
2. Click the current kernel name (it may say "Select Kernel", "No Kernel", or "Python 3").
3. Select the active **Python 3.11** environment native to your DevContainer or CoChem workspace.

**Step 2: Start the Orchestrator**
1. Click into **Cell 2** below.
2. Press `Shift + Enter` to execute. 
3. *Note: The very first output will print your `sys.executable` path to instantly verify your kernel telemetry is correct before the 7-phase build begins.*

In [24]:
import sys
import subprocess
from pathlib import Path

print(f"✅ Kernel Connected: {sys.executable}\n")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'stage_0_0_setup_orchestrator.py').exists():
    candidate = REPO_ROOT / 'CoChem-CORE'
    if (candidate / 'stage_0_0_setup_orchestrator.py').exists():
        REPO_ROOT = candidate

cmd = [sys.executable, str(REPO_ROOT / 'stage_0_0_setup_orchestrator.py')]
print('⚙️ Booting Orchestrator:', ' '.join(cmd))
print('-' * 60)

result = subprocess.run(cmd, cwd=str(REPO_ROOT), check=False)
if result.returncode != 0:
    raise RuntimeError(f'❌ stage_0_0_setup_orchestrator.py failed with exit code {result.returncode}')

print('-' * 60)
print('✅ CoChem-CORE setup completed successfully.')


✅ Kernel Connected: /home/joshua/anaconda3/envs/cochem_torq_silo/bin/python

⚙️ Booting Orchestrator: /home/joshua/anaconda3/envs/cochem_torq_silo/bin/python /media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/stage_0_0_setup_orchestrator.py
------------------------------------------------------------
 CoChem Pipeline Initialization Orchestrator 

📂 Repository Root: /media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE
🔒 Air-Gap Enforced: All logs and state files routed to /home/joshua/CoChem_Artifacts
📦 Engine Staging Directory locked at: /home/joshua/CoChem_Artifacts/Registry/Engines
📍 Place ORCA archives in this exact folder before rerunning setup if prompted.

🚀 Initiating CoChem Core Bootstrapper...

▶️ Executing cochem_setup_phase_1.py...

 CoChem Phase 1: Core System Auditing & Profiling 

🧹 Purged ghost dependencies (LD_LIBRARY_PATH) to ensure silo sterility.
📈 OS Stack Size tuned to maximum limit (-1) for heavy Tensor evaluations.
✅ Essential C++ an

/usr/bin/bash: /home/joshua/anaconda3/lib/libtinfo.so.6: no version information available (required by /usr/bin/bash)


# CoChem-MInt Molecule Intake
Run the next cell to load the molecule intake widget. Add your `.xyz` files to `CoChem_Artifacts/Input_Files`, then use the widget controls to scan/build/watch.

In [16]:
# cochem_canvas_target: Start_Here_MInt_Cell.py
# %% [markdown]
# ## Stage 1: CoChem-MInt (Molecular Intake & Validation)
# Executes the interactive IPyWidgets dashboard for PubChem fetch or local `.xyz` ingestion.

# %%
import sys
import importlib.util
from pathlib import Path

# Safely anchor the REPO_ROOT to the current working directory of the notebook
REPO_ROOT = Path.cwd()
mint_script = REPO_ROOT / 'intake' / 'CoChem-MInt.py'

if not mint_script.exists():
    raise FileNotFoundError(f'Missing CoChem-MInt script at: {mint_script}')

spec = importlib.util.spec_from_file_location('cochem_mint', str(mint_script))
mint_module = importlib.util.module_from_spec(spec)
# Inject REPO_ROOT into the module's namespace so it doesn't fail on relative imports
mint_module.REPO_ROOT = REPO_ROOT
sys.modules['cochem_mint'] = mint_module
spec.loader.exec_module(mint_module)

mint_ui = mint_module.CoChemMIntUI()
mint_ui.display()


# Clone CoChem-TOPOS and CoChem-TORQ
This cell attempts to clone both repositories next to `CoChem-CORE` in the same parent workspace directory.

In [17]:
workspace_root = REPO_ROOT.parent
repos = {
    'CoChem-TOPOS': 'https://github.com/CoChem/CoChem-TOPOS',
    'CoChem-TORQ': 'https://github.com/CoChem/CoChem-TORQ',
}

for name, url in repos.items():
    target = workspace_root / name
    if target.exists():
        print(f'Already exists: {target}')
        continue

    probe = subprocess.run(['git', 'ls-remote', '--heads', url], capture_output=True, text=True, check=False)
    if probe.returncode != 0:
        alt_url = f'{url}.git'
        probe_alt = subprocess.run(['git', 'ls-remote', '--heads', alt_url], capture_output=True, text=True, check=False)
        if probe_alt.returncode != 0:
            print(f'Clone skipped for {name}: repository not reachable from this environment.')
            print((probe.stderr or probe_alt.stderr or '').strip())
            continue
        url = alt_url

    clone = subprocess.run(['git', 'clone', '--depth', '1', url, str(target)], check=False)
    if clone.returncode == 0:
        print(f'Cloned {name} -> {target}')
    else:
        print(f'Clone failed for {name} (exit {clone.returncode})')


Already exists: /media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-TOPOS
Already exists: /media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-TORQ


# Configure Manifest and Build Silos
This cell writes a manifest selecting `CoChem-CORE`, `CoChem-TOPOS`, and `CoChem-TORQ`, then runs Phase 4 and Phase 5 to ensure silos are prepared.

In [23]:
import json

setup_dir = REPO_ROOT / 'cochem_setup'
setup_dir.mkdir(parents=True, exist_ok=True)
manifest_path = setup_dir / 'cochem_deployment_manifest.json'

manifest = {
    'version': '2026.2',
    'deployment_target': 'Local DevContainer',
    'modules': ['CoChem-CORE', 'CoChem-TOPOS', 'CoChem-TORQ'],
    'selected_modules': {
        'CoChem-CORE': 'https://github.com/CoChem/CoChem-CORE',
        'CoChem-TOPOS': 'https://github.com/CoChem/CoChem-TOPOS',
        'CoChem-TORQ': 'https://github.com/CoChem/CoChem-TORQ'
    },
    'host_orca_path': ''
}

manifest_path.write_text(json.dumps(manifest, indent=4), encoding='utf-8')
print(f'Manifest written: {manifest_path}')

for phase in ['cochem_setup_phase_4.py', 'cochem_setup_phase_5.py']:
    phase_path = setup_dir / phase
    cmd = [sys.executable, str(phase_path)]
    print('Running:', ' '.join(cmd))
    rc = subprocess.run(cmd, cwd=str(REPO_ROOT), check=False).returncode
    if rc != 0:
        raise RuntimeError(f'{phase} failed with exit code {rc}')

print('Silo setup phases completed.')


Manifest written: /media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_deployment_manifest.json
Running: /home/joshua/anaconda3/envs/cochem_torq_silo/bin/python /media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_setup_phase_4.py

--- CoChem Phase 4: Micro-Silo Orchestration ---
 ➡️ Probing for Conda/Mamba package manager...
 ✅ Package Manager Detected: /home/joshua/anaconda3/condabin/conda
 ➡️ Building TOPOS Silo 'cochem_topos_silo' (Python 3.11, RDKit, NetworkX)...
 ➡️ Enforcing pip/wheel upgrade in silo 'cochem_topos_silo'...
 ➡️ Injecting Jupyter dependencies for TOPOS...
 ➡️ Registering TOPOS Silo into Jupyter environment kernels...
 ✅ TOPOS Silo built and registered successfully.
 ➡️ Building TORQ Silo 'cochem_torq_silo' (Python 3.11, PyTorch, MACE)...
 ➡️ Enforcing pip/wheel upgrade in silo 'cochem_torq_silo'...


Traceback (most recent call last):
  File "/media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_setup_phase_4.py", line 405, in <module>
    main()
  File "/media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_setup_phase_4.py", line 368, in main
    if build_torq_silo(conda_path, log):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_setup_phase_4.py", line 273, in build_torq_silo
    enforce_pip_upgrades(conda_path, env_name, log)
  File "/media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_setup_phase_4.py", line 179, in enforce_pip_upgrades
    return run_conda_cmd(
           ^^^^^^^^^^^^^^
  File "/media/joshua/Storage/GitHub-Repo/CoChem-SYNAP/modules/CoChem-CORE/cochem_setup/cochem_setup_phase_4.py", line 140, in run_conda_cmd
    result = subprocess.run(
             ^^^^^^^^^^^^^^^
  File "/ho

KeyboardInterrupt: 

# Run Setup for Cloned Modules
This cell runs each cloned module orchestrator if present to complete module-specific setup.

In [19]:
module_dirs = [REPO_ROOT.parent / 'CoChem-TOPOS', REPO_ROOT.parent / 'CoChem-TORQ']
for module_dir in module_dirs:
    if not module_dir.exists():
        print(f'Skipping missing module directory: {module_dir}')
        continue

    orchestrator = module_dir / 'stage_0_0_setup_orchestrator.py'
    if not orchestrator.exists():
        print(f'No setup orchestrator found in {module_dir.name}; skipping module setup.')
        continue

    cmd = [sys.executable, str(orchestrator)]
    print(f"Running {module_dir.name} setup: {' '.join(cmd)}")
    rc = subprocess.run(cmd, cwd=str(module_dir), check=False).returncode
    if rc != 0:
        raise RuntimeError(f'{module_dir.name} setup failed with exit code {rc}')

print('TOPOS/TORQ module setup pass complete.')


No setup orchestrator found in CoChem-TOPOS; skipping module setup.
No setup orchestrator found in CoChem-TORQ; skipping module setup.
TOPOS/TORQ module setup pass complete.


# Optional: Show Recent Logs

In [20]:
artifact_dir = Path(os.environ.get('COCHEM_ARTIFACT_DIR', str(Path.home() / 'CoChem_Artifacts')))
log_dir = artifact_dir / 'Logs'
print(f'Log directory: {log_dir}')

if not log_dir.exists():
    print('No logs found yet.')
else:
    for path in sorted(log_dir.glob('*.log')):
        print(f'\n=== {path.name} ===')
        lines = path.read_text(errors='replace').splitlines()
        print('\n'.join(lines[-40:]) if lines else '<empty>')


NameError: name 'os' is not defined

In [21]:
from IPython.display import display

dashboard_script = REPO_ROOT / 'interfaces' / 'cochem_unity_installer_dashboard.py'
if not dashboard_script.exists():
    raise FileNotFoundError(f'Missing UNITY dashboard script: {dashboard_script}')

dashboard_spec = importlib.util.spec_from_file_location('cochem_unity_installer_dashboard', str(dashboard_script))
dashboard_module = importlib.util.module_from_spec(dashboard_spec)
dashboard_spec.loader.exec_module(dashboard_module)

installer = dashboard_module.UnityInstallerGUI()
display(installer.main_ui)
